# 第1章 はじめに
## 1.1 transformersを使って自然言語処理を解いてみよう

In [1]:
from transformers import pipeline

/Users/ysumita/work/python/llm-book/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 文書分類
text_classification_pipeline = pipeline(
    model="llm-book/bert-base-japanese-v3-marc_ja"
)

positive_text = "世界には言葉がわからなくても感動する音楽がある。"
print(text_classification_pipeline(positive_text)[0])

negative_text = "世界には言葉がでないほどひどい音楽がある。"
print(text_classification_pipeline(negative_text)[0])

/Users/ysumita/work/python/llm-book/.venv/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


{'label': 'positive', 'score': 0.9993619322776794}
{'label': 'negative', 'score': 0.9636250734329224}


In [3]:
# 自然言語推論
nli_pipeline = pipeline(model="llm-book/bert-base-japanese-v3-jnli")

text = "二人の男性がジェット機を見ています"

entailment_text = "ジェット機を見ている人が二人います"
print(nli_pipeline({"text": text, "text_pair": entailment_text}))

contradiction_text = "二人の男性が飛んでいます"
print(nli_pipeline({"text": text, "text_pair": contradiction_text}))

neutral_text = "２人の男性が、白い飛行機を眺めています"
print(nli_pipeline({"text": text, "text_pair": neutral_text}))

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


{'label': 'entailment', 'score': 0.9964311122894287}
{'label': 'contradiction', 'score': 0.9990535378456116}
{'label': 'neutral', 'score': 0.9959145188331604}


In [4]:
# 意味的類似度計算
text_sim_pipeline = pipeline(
    model="llm-book/bert-base-japanese-v3-jsts",
    function_to_apply="none",
)

text = "川べりでサーフボードを持った人たちがいます"

sim_text = "サーファーたちが川べりに立っています"
result = text_sim_pipeline({"text": text, "text_pair": sim_text})
print(result["score"])

dissim_text = "トイレの壁に黒いタオルがかけられています"
result = text_sim_pipeline({"text": text, "text_pair": dissim_text})
print(result["score"])

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


3.5703563690185547
0.041621237993240356


In [ ]:
## 文埋め込みを使う
from torch.nn.functional import cosine_similarity

sim_enc_pipeline = pipeline(
    model="llm-book/bert-base-japanese-v3-unsup-simcse-jawiki",
    task="feature-extraction",
)

text_emb = sim_enc_pipeline(text, return_tensors=True)[0][0]

sim_emb = sim_enc_pipeline(sim_text, return_tensors=True)[0][0]
sim_pair_score = cosine_similarity(text_emb, sim_emb, dim=0)
print(sim_pair_score.item())

dissim_emb = sim_enc_pipeline(dissim_text, return_tensors=True)[0][0]
dissim_pair_score = cosine_similarity(text_emb, dissim_emb, dim=0)
print(dissim_pair_score.item())

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


0.8568587899208069
0.458870530128479


In [6]:
# 固有表現認識
from pprint import pprint

ner_pipeline = pipeline(
    model="llm-book/bert-base-japanese-v3-ner-wikipedia-dataset",
    aggregation_strategy="simple",
)
text = "大谷翔平は岩手県水沢市出身のプロ野球選手"
pprint(ner_pipeline(text))

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


[{'end': None,
  'entity_group': '人名',
  'score': np.float32(0.99823624),
  'start': None,
  'word': '大谷 翔平'},
 {'end': None,
  'entity_group': '地名',
  'score': np.float32(0.9986874),
  'start': None,
  'word': '岩手 県 水沢 市'}]


In [7]:
# 要約生成
text2text_pipeline = pipeline(
    model="llm-book/t5-base-long-livedoor-news-corpus"
)
article = "ついに始まった３連休。テレビを見ながら過ごしている人も多いのではないだろうか？　今夜オススメなのは何と言っても、NHKスペシャル「世界を変えた男 スティーブ・ジョブズ」だ。実は知らない人も多いジョブズ氏の養子に出された生い立ちや、アップル社から一時追放されるなどの経験。そして、彼が追い求めた理想の未来とはなんだったのか、ファンならずとも気になる内容になっている。 今年、亡くなったジョブズ氏の伝記は日本でもベストセラーになっている。今後もアップル製品だけでなく、世界でのジョブズ氏の影響は大きいだろうと想像される。ジョブズ氏のことをあまり知らないという人もこの機会にぜひチェックしてみよう。 世界を変えた男　スティーブ・ジョブズ（NHKスペシャル）"
print(text2text_pipeline(article)[0]["generated_text"])

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.
/Users/ysumita/work/python/llm-book/.venv/lib/python3.10/site-packages/transformers/generation/utils.py:1258: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


今夜はNHKスペシャル「世界を変えた男 スティーブ・ジョブズ」をチェック!


## 1.2 transformersの基本的な使い方

In [8]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("abeja/gpt2-large-japanese")
tokenizer.tokenize("今日は天気が良いので")

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


['▁', '今日', 'は', '天気', 'が良い', 'の', 'で']

In [9]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    "abeja/gpt2-large-japanese",
    device_map="auto"
)
inputs = tokenizer("今日は天気が良いので", return_tensors="pt")
outputs = model.generate(
    **inputs,
    max_length=15,
    pad_token_id=tokenizer.pad_token_id
)
generated_text = tokenizer.decode(
    outputs[0], skip_special_tokens=True
)
print(generated_text)

/Users/ysumita/work/python/llm-book/.venv/lib/python3.10/site-packages/transformers/generation/utils.py:1885: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on mps. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('mps') before running `.generate()`.
  warnings.warn(


RuntimeError: Placeholder storage has not been allocated on MPS device!